<div align="center">
  <a href="https://colab.research.google.com/github/PrunaAI/ai-efficiency-courses/blob/main/solutions/02-run_llm_cpu_vs_gpu.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
  </a>
</div>

---
**💡 Tip**: Click the button above to open this notebook in Google Colab for free GPU access!

## Installation

This notebook includes automatic setup cells that will install the project from git repository with UV.

**Note**: Run the setup cells below before starting the exercises.

In [ ]:
# Install project directly from git repository
!uv pip install git+https://github.com/PrunaAI/ai-efficiency-courses.git

## Utility cells

During the course, we'll leverage some course utilities to streamline our workflow. These utilities are located in the `course` package, which can simply be imported given that we installed the project from git repository above.
You can find the source code [here](https://github.com/PrunaAI/ai-efficiency-courses/tree/main/course).

These utilities will help us:
- Load and manage lists of model ids that we have verified to work.
- Generate informative plots for model analysis.
- Iterate efficiently over evaluation and model configuration options.

Let's first load our models. We will use `SMALL_MODEL_IDS`, which are sub 1B parameters which should be easy to download and load into memory. We recommend starting with these smaller models but feel free to experiment with other models until you reach your GPU memory limit!

In [ ]:
from course import SMALL_MODEL_IDS, MEDIUM_MODEL_IDS, LARGE_MODEL_IDS, ALL_MODEL_IDS

MODEL_IDS = SMALL_MODEL_IDS
# MODEL_IDS = MEDIUM_MODEL_IDS
# MODEL_IDS = LARGE_MODEL_IDS
# MODEL_IDS = ALL_MODEL_IDS

MODEL_IDS

We also recommend to set a custom cache directory for models. Loading models can take significant disk space. To avoid filling up your default disk, we recommend setting a custom cache directory for downloaded models. You can do this by running the following in a terminal or in a notebook cell:

In [ ]:
# Replace <path_to_cache> with your desired cache path
import os

CACHE_PATH = "<path_to_cache>"
os.environ["TORCH_HOME"] = CACHE_PATH
os.environ["HF_HOME"] = CACHE_PATH
os.environ["HUGGINGFACE_HUB_CACHE"] = CACHE_PATH
os.environ["HUGGINGFACE_ASSETS_CACHE"] = CACHE_PATH

You can also clear the cache by running the following cell:

In [ ]:
from course.models import clear_cache

clear_cache(CACHE_PATH)

# 02: Run LLMs on CPU vs GPU

Welcome to this new lecture of the AI Efficiency course! 🚀

In this tutorial, we will analyze the pros and cons to run Large Language Models (LLMs) on CPUs vs GPUs. It is particularly important to take informed hardware decisions since they can have big impact on inference time, and money costs. The content from the chapter 2 [slides](slides/02-compress_language_models.pdf) will help you to go through this notebook.

By the end of this lecture, you will:
- Understand how to run LLMs on CPUs and GPUs.
- Be able to compare the efficiency of LLMs on CPU vs GPU.
- Learn how to improve efficiency of LLMs on CPU and GPU.

Let's get started on running LLMs on different hardware!

## 1. Imports

As we've already installed the project, we can import the necessary libraries. We will be using `torch` and `transformers` for this tutorial as interfaces to the model and tokenizer. On top of that, we will be using `pruna` to run the evaluation and `matplotlib` for basic plotting. We recommend to checkout the [Pruna documentation](https://docs.pruna.ai/en/stable/docs_pruna/user_manual/evaluation.html) for access to AI efficiency functions.

In [3]:
import copy
import gc

import torch
from pruna import PrunaModel, SmashConfig, smash
from pruna.data.pruna_datamodule import PrunaDataModule
from pruna.evaluation.evaluation_agent import EvaluationAgent
from pruna.evaluation.metrics import (
    LatencyMetric,
)
from pruna.evaluation.task import Task
from transformers import AutoModelForCausalLM, AutoTokenizer

Beyond external libraries, this course comes with the `course` local package which contains a lot of utils that you can use in the notebooks. We will be using `create_single_plot` to create a simple plot.

In [4]:
from course import create_single_plot

## 2. Run LLMs on CPU vs GPU

We recommend to check the [evaluation guide](https://docs.pruna.ai/en/stable/docs_pruna/user_manual/evaluate.html) and [metrics overview](https://docs.pruna.ai/en/stable/reference/evaluation.html#metrics-overview) in the pruna documentation for the implementation details.

### 2.1 Run base LLM on CPU vs GPU

In this section, you'll learn how to benchmark and compare LLM performance across CPU and GPU hardware.

**Why is this important?**
Understanding how LLMs perform on different hardware helps you make informed deployment decisions. The choice between CPU and GPU can significantly impact latency, throughput, and cost-effectiveness of your LLM applications.

**Your tasks:**
1. Create the `evaluate_models` function to benchmark the models, which loads the input model ids and performs the evaluation.
2. Benchmark base models on CPU by measuring the latency metrics for the base models running on CPU hardware.
3. Benchmark base models on GPU by measuring the same latency metrics but with GPU acceleration.

**Key questions to consider:**
- How do CPU vs GPU speeds compare? What happens with larger batch sizes?
- What scenarios favor CPU vs GPU deployment?
- Beyond latency, what other metrics (quality, memory, compute) would be valuable to compare?
- How do hardware choices impact real-world deployment considerations?

As you complete this section, reflect on how hardware selection influences the practical deployment and scaling of LLM applications.

In [ ]:
def evaluate_models(model_ids: list, metrics: list, dataset="WikiText") -> dict:
    """
    Evaluate multiple models using the specified metrics.

    Args:
        model_ids (list): List of model IDs to evaluate
        metrics (list): List of metric instances to compute

    Returns:
        dict: Dictionary mapping model IDs to their evaluation results
    """
    results = {}

    ### To Complete ###

Now let's run the base models on CPU...

In [ ]:
metrics = [
    LatencyMetric(
        n_iterations=10,
        n_warmup_iterations=10,
        device="cpu",
        timing_type="sync",
    ),
]

### To Complete ###

... and let's check now the performanc in a GPU.

In [ ]:
### To Complete ###

### 2.2 Run quantized LLM on CPU vs GPU

In this section, you'll learn how to systematically quantize and benchmark LLMs on different hardware platforms. We recommend to check the quantization features in [pruna documentation](https://docs.pruna.ai/en/stable/compression.html#quantizers) for this.

**Why is this important?**
Understanding how quantization affects model performance across CPU and GPU helps optimize deployment. Quantization reduces model size and can improve inference speed, but the benefits vary by hardware. Making informed quantization choices is crucial for efficient real-world applications.

**Your tasks:**
1. Apply CPU-specific quantization (e.g., IPEX-LLM) and measure performance with an odd context length.
2. Apply GPU-specific quantization (e.g., LLM-INT8) and evaluate performance.

**Key questions to answer:**
- What quantization methods are used for CPU vs GPU? How do they differ?
- How does quantization impact model speed compared to the base versions?
- What hardware-quantization combinations work best for:
  - Real-time interactive applications?
  - Edge device deployment?
  - Batch processing workloads?

As you complete this section, reflect on how quantization choices interact with hardware selection to influence practical deployment scenarios.

In [9]:
# Select the model and load it
model_id = MODEL_IDS[0]
model = AutoModelForCausalLM.from_pretrained(model_id)
tokenizer = AutoTokenizer.from_pretrained(model_id)

In [ ]:
# Quantize on CPU
smash_config = SmashConfig()
### To Complete ###

In [ ]:
# Quantize on GPU
smash_config = SmashConfig()
### To Complete ###

## Conclusion: What We've Learned About LLM on CPU and GPU

In this module, we explored running LLMs on different hardware configurations and compared their performance characteristics. Here are the key findings:

- **GPU vs CPU Speed:**
  Running LLMs on GPU provides significantly faster inference - typically an order of magnitude faster than CPU (even after compression). This makes GPUs the preferred choice for production deployments where latency is critical.

- **Memory Considerations:**
  While GPUs offer superior speed, CPU deployments can be advantageous for memory-constrained scenarios:
  - CPU memory (RAM) is generally much cheaper, flexible, and abundant than GPU VRAM
  - Edge devices and smaller servers often lack GPUs but have sufficient CPU resources

- **Deployment Tradeoffs:**
  The choice between CPU and GPU depends on your specific needs:
  - Use GPU when speed is critical and you have access to the hardware
  - Consider CPU for cost-sensitive deployments or edge scenarios
  - Hybrid approaches may work best for some applications

### Next Steps: Measure LLM Efficiency

Now that you understand the performance characteristics of different hardware configurations, you can make informed decisions about deployment architecture. The next sections will explore how to measure the efficiency of LLMs in detail.

👉 **Continue to the next notebook:**

[03-measure_llm_efficiency.ipynb on GitHub](https://github.com/PrunaAI/ai-efficiency-courses/blob/main/exercises/03-measure_llm_efficiency.ipynb)

## ⭐ Bonus Exercise: Evaluating LLM Performance Across Hardware

As a bonus, try comparing model performance between CPU and GPU. Hardware choice can dramatically impact inference speed, energy usage, and memory requirements—see if you can quantify these tradeoffs!

**Your tasks:**
1. **Compare inference metrics across hardware:**
   Analyze and compare key performance metrics between CPU and GPU execution.
2. **Evaluate quality vs efficiency tradeoffs:**
   Study how hardware choice affects the balance between model quality and resource efficiency.

**Key questions to consider:**
- What are the key differences in performance between CPU and GPU execution?
- How do memory usage patterns differ between the two hardware options?
- What are the energy efficiency tradeoffs between CPU and GPU inference?